In [1]:
!pip install chromadb sentence-transformers pandas tqdm

In [2]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [3]:
combined_df = pd.read_csv("processed_dhl_dataset.csv")

print("Dataset Shape:", combined_df.shape)

combined_df.head()

Dataset Shape: (308, 5)


,title,content,url,source_name,clean_text
0,Great Place To Work (R) Institute Japan「働きがいのあ...,実際に働く従業員の声をもとに「働きがい」の高い企業を認定2026年6月26日Great Pl...,https://japan.cnet.com/release/31182001/,NewsAPI,great place work r institute japan 2026 5 2026...
1,FedEx to Return $800 Million in Tariff Refunds,FedEx will return approximately $800 million i...,http://wwd.com/sourcing-journal/logistics/fede...,NewsAPI,fedex return 800 million tariff refunds fedex ...
2,FedEx to Return $800 Million in Tariff Refunds,"Reimbursements will begin in August, says the ...",https://wwd.com/sourcing-journal/logistics/fed...,NewsAPI,fedex return 800 million tariff refunds reimbu...
3,Postal Service says its cash crisis is delayed...,The U.S. Postal Service is no longer set to be...,https://www.npr.org/2026/06/24/nx-s1-5859334/u...,NewsAPI,postal service says cash crisis delayed least ...
4,NAHCO secures fresh airline contracts,"Nigerian Aviation Handling Company Plc, NAHCO ...",https://www.vanguardngr.com/2026/06/nahco-secu...,NewsAPI,nahco secures fresh airline contracts nigerian...


In [4]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        307 non-null    object
 1   content      307 non-null    object
 2   url          266 non-null    object
 3   source_name  308 non-null    object
 4   clean_text   308 non-null    object
dtypes: object(5)
memory usage: 12.2+ KB


In [5]:
combined_df["clean_text"] = combined_df["clean_text"].fillna("")

combined_df["clean_text"] = combined_df["clean_text"].astype(str)

In [6]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [7]:
texts = combined_df["clean_text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print("Embedding Shape:", embeddings.shape)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding Shape: (308, 384)


In [8]:
CHROMA_PATH = "dhl_chromadb"

client = chromadb.PersistentClient(path=CHROMA_PATH)

In [9]:
try:
    client.delete_collection("dhl_intelligence")
    print("Old collection deleted.")
except:
    print("No existing collection found.")

Old collection deleted.


In [10]:
collection = client.create_collection(
    name="dhl_intelligence"
)

print("Collection Created Successfully!")

Collection Created Successfully!


In [11]:
from tqdm import tqdm

for i in tqdm(range(len(combined_df))):

    collection.add(
        ids=[str(i)],
        embeddings=[embeddings[i].tolist()],
        documents=[combined_df.iloc[i]["clean_text"]],
        metadatas=[{
            "title": str(combined_df.iloc[i]["title"]),
            "source": str(combined_df.iloc[i]["source_name"])
        }]
    )

print("All documents stored successfully!")

100%|██████████| 308/308 [00:12<00:00, 23.95it/s]

All documents stored successfully!


In [12]:
print("Total Documents in ChromaDB:")

print(collection.count())

Total Documents in ChromaDB:
308


In [13]:
query = "AI opportunities in DHL"

results = collection.query(
    query_texts=[query],
    n_results=5
)

In [14]:
print("Top 5 Retrieved Documents:\n")

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"Document {i}")
    print(doc[:300])
    print("-" * 80)

Top 5 Retrieved Documents:

Document 1
dhl integrates happyrobot ai agents streamline logistics dhl supply chain partnering happyrobot roll autonomous ai agents handle high volume communication streamline scheduling improve warehouse coordination enhance customer service initiative part dhl broader ai strategy boost efficiency employee e
--------------------------------------------------------------------------------
Document 2
nov 11 2025 dhl boosts operational efficiency customer communications happyrobot ai agents dhl group november 11 2025 part structured strategic approach ai dhl supply chain systematically identifying validating operational use cases generative agentic ai technologies 18 months building extensive ope
--------------------------------------------------------------------------------
Document 3
dhl ai agent bet logistics ops leaders debales ai ready see ai agents handle appointment scheduling check calls email automation way dhl without 18 month build timeline
------